In [41]:
# ---------------------------------------------------------
# STEP 1: INSTALL REQUIRED PACKAGES
# ---------------------------------------------------------

# Install LangGraph for building the multi-agent workflow.
!pip install -U langgraph

# Install LangChain for working with language models.
!pip install -U langchain

# Install the Google Gemini integration for LangChain.
!pip install -U langchain-google-genai

# Install python-dotenv for loading our Gemini API key
# from the .env file.
!pip install -U python-dotenv

# Print a message when installation is complete.
print("All required packages have been installed!")

All required packages have been installed!


In [42]:
# ---------------------------------------------------------
# STEP 2: IMPORT REQUIRED PACKAGES
# ---------------------------------------------------------

# Import LangGraph.
import langgraph

# Import LangChain.
import langchain

# Import the Gemini chat model.
from langchain_google_genai import ChatGoogleGenerativeAI

# Import dotenv for loading environment variables.
from dotenv import load_dotenv

# Import os for accessing environment variables.
import os

# Print the LangChain version.
print("LangChain version:", langchain.__version__)

# Confirm that LangGraph loaded.
print("LangGraph imported successfully!")

# Confirm that Gemini integration loaded.
print("Gemini imported successfully!")

# Confirm that dotenv loaded.
print("python-dotenv imported successfully!")

LangChain version: 1.4.1
LangGraph imported successfully!
Gemini imported successfully!
python-dotenv imported successfully!


In [43]:
# ---------------------------------------------------------
# STEP 4: LOAD AND TEST THE GEMINI API KEY
# ---------------------------------------------------------

# Load the variables stored inside the .env file.
load_dotenv()

# Get the Gemini API key from the environment.
api_key = os.getenv("GEMINI_API_KEY")

# Check whether the API key was found.
if api_key:
    
    # Display a success message.
    print("Gemini API key loaded successfully!")

else:
    
    # Display an error message if the key was not found.
    print("Gemini API key NOT found.")

Gemini API key loaded successfully!


In [44]:
# ---------------------------------------------------------
# STEP 5: CREATE THE GEMINI AI MODEL
# ---------------------------------------------------------

# Import the Gemini chat model from LangChain.
from langchain_google_genai import ChatGoogleGenerativeAI

# Create our Gemini model.
# This model will be used by all of our agents.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

# Confirm that the model object was created.
print("Gemini AI model created successfully!")

Gemini AI model created successfully!


In [45]:
# ---------------------------------------------------------
# STEP 6: TEST GEMINI
# ---------------------------------------------------------

# Send a simple question to Gemini.
response = llm.invoke(
    "Explain artificial intelligence in one simple sentence."
)

# Display Gemini's response.
print(response.content)

[{'type': 'text', 'text': 'Artificial intelligence is technology that allows computers to learn, think, and solve problems much like a human mind.', 'extras': {'signature': 'EvIPCu8PAWkUfRM5fGpiHvr0uSSUi9HJ+jbq9UotLh3Zmj46htz2VTOQtEr8wgIIS07qRCPi+os7dG9J+MXgPak1xHCcUn2bbYPpq3VFi4N2yAqFnkWouROA3j3fnXsvEiTfQe256rklXPvQcWpynXmYTXgjUTn1OvUQXZXj+uMS3k4/0CKtbiSgSxlPHQTqfYpAbrxZjdD5cZTDw0NjRfnmQS8+6sQ57oEQY0NG9NQjzhsZ40Yk4tYtluDLGuTJJhKISkPE4HwNkPDegRIt6aRD7pLCzFhCmUVZyev1QICk8RwL93Ui58v/GBbx84x1dYbOsvJgl/eLF3Sy5D1yfP8TEv7Emyv08/1dtzVNWrpWjB1QMKJCBozEcze53hU4OQa72Fsm9goUYyQfoxdke7vW9c1chz3MMWiBMt9EQxus9iui70F9R8cLKKaB/kwBC8kK60kDJVVUXLKSFmHKDy43vA5qQSf3g8xUMKplr7f4iJqtDvegj4F9r03uy5QekuPWMWLK/eV674dK+4RbTSHWh7YRLucjy3BARw4B/7WG3cwm23yW+UsDrBukmYwybwrtL1xXE4ZMDaG/vzWGwmjtA+OQShCrZ5ZejeDxLaVEvWQu2RhQJ0Dqik+nBqw5zVk2ciruZ1OEcOTFZ8h0/VJlLN4YldRQ9cqKKQBN1VAepdWX1+JNiP67tB9V+esVXAJwEFgCMUZ5CsfOWiLZlU7mDpOXBq1cY7GBi4mpvEOtna0k0yTz+kuoH+ZVNSx64Z3eHVKLLCU72zqAwOMRvAQ+r5zQbRn76uyKzsdmcY9ct6VRysoIu2LLjA

In [46]:
# ---------------------------------------------------------
# STEP 7: CREATE THE SHARED WORKFLOW STATE
# ---------------------------------------------------------

# TypedDict allows us to define the structure
# of information shared between our agents.
from typing import TypedDict


# Create the shared state for our research system.
class ResearchState(TypedDict):

    # The original question from the user.
    question: str

    # Information collected by the Research Agent.
    research: str

    # Information produced by the Fact Checker Agent.
    fact_check: str

    # Conclusions produced by the Analysis Agent.
    analysis: str

    # Final report produced by the Writer Agent.
    final_report: str

    # The Supervisor's decision about the next agent.
    next_agent: str


# Confirm that the shared state was created.
print("ResearchState created successfully!")

ResearchState created successfully!


In [47]:
# ---------------------------------------------------------
# STEP 8: CREATE THE RESEARCH AGENT
# ---------------------------------------------------------

# Define the Research Agent as a Python function.
def research_agent(state: ResearchState):

    # Get the user's question from the shared state.
    question = state["question"]

    # Create instructions for Gemini.
    prompt = f"""
You are the Research Agent in a multi-agent research system.

Research the following question:

{question}

Provide:

1. Key facts
2. Important explanations
3. Relevant details
4. Useful examples

Keep the information clear and concise.

Do not write the final report.
Your job is to provide research findings for the next agents.
"""

    # Send the research instructions to Gemini.
    response = llm.invoke(prompt)

    # Store the research result in the shared state.
    return {
        "research": response.content
    }


# Confirm that the Research Agent was created.
print("Research Agent created successfully!")

Research Agent created successfully!


In [49]:
# ---------------------------------------------------------
# STEP 9: TEST THE RESEARCH AGENT
# ---------------------------------------------------------

# Create a small test state.
test_state = {

    # Give the agent a research question.
    "question": "What are the applications of generative AI in education?",

    # Start the other fields as empty.
    "research": "",
    "fact_check": "",
    "analysis": "",
    "final_report": "",
    "next_agent": ""
}


# Run the Research Agent using the test state.
research_result = research_agent(test_state)


# Display the research produced by the agent.
print("===== RESEARCH RESULT =====")
print(research_result["research"])

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 48.369723781s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [50]:
# ---------------------------------------------------------
# STEP 10: IMPORT LANGGRAPH COMPONENTS
# ---------------------------------------------------------

# StateGraph is used to create our workflow.
from langgraph.graph import StateGraph, START, END

# Confirm that the LangGraph components loaded.
print("LangGraph components imported successfully!")

LangGraph components imported successfully!


In [51]:
# ---------------------------------------------------------
# STEP 11: CREATE THE FACT CHECKER AGENT
# ---------------------------------------------------------

# Define the Fact Checker Agent.
def fact_checker_agent(state: ResearchState):

    # Get the research from the shared workflow state.
    research = str(state["research"])[:6000]

    # Create a prompt for Gemini.
    prompt = f"""
You are the Fact Checker Agent.

Review the research below:

{research}

Identify:
1. Reliable-looking claims
2. Claims that need verification
3. Possible inaccuracies
4. Important limitations

Keep the response concise.
Do not write the final report.
"""

    # Ask Gemini to perform the fact-checking.
    response = llm.invoke(prompt)

    # Return the fact-checking result.
    return {
        "fact_check": str(response.content)
    }


# Confirm that the function was created.
print("Fact Checker Agent created successfully!")

Fact Checker Agent created successfully!


In [52]:
# ---------------------------------------------------------
# STEP 12: CREATE THE ANALYSIS AGENT
# ---------------------------------------------------------

# Define the Analysis Agent.
def analysis_agent(state: ResearchState):

    # Get the research from the shared state.
    research = str(state["research"])[:6000]

    # Get the fact-checking information.
    fact_check = str(state["fact_check"])[:4000]

    # Create a prompt for Gemini.
    prompt = f"""
You are the Analysis Agent.

Analyze the following research and fact-checking information.

RESEARCH:
{research}

FACT CHECK:
{fact_check}

Provide:
1. Main insight
2. Important conclusion
3. Practical implication
4. Limitation

Keep the analysis concise.
Do not write the final report.
"""

    # Ask Gemini to perform the analysis.
    response = llm.invoke(prompt)

    # Return the analysis result.
    return {
        "analysis": str(response.content)
    }


# Confirm that the function was created.
print("Analysis Agent created successfully!")

Analysis Agent created successfully!


In [53]:
# ---------------------------------------------------------
# STEP 13: CREATE THE WRITER AGENT
# ---------------------------------------------------------

# Define the Writer Agent.
def writer_agent(state: ResearchState):

    # Get the original question.
    question = state["question"]

    # Get the research information.
    research = str(state["research"])[:5000]

    # Get the fact-checking information.
    fact_check = str(state["fact_check"])[:3000]

    # Get the analysis.
    analysis = str(state["analysis"])[:3000]

    # Create instructions for Gemini.
    prompt = f"""
You are the Writer Agent in a multi-agent research system.

Write a clear final report answering the user's question.

QUESTION:
{question}

RESEARCH:
{research}

FACT CHECK:
{fact_check}

ANALYSIS:
{analysis}

Create the final report with:

1. Title
2. Introduction
3. Main findings
4. Analysis
5. Conclusion

Use clear and simple language.
Do not mention the internal agents or workflow.
"""

    # Ask Gemini to write the final report.
    response = llm.invoke(prompt)

    # Return the final report.
    return {
        "final_report": str(response.content)
    }


# Confirm that the Writer Agent was created.
print("Writer Agent created successfully!")

Writer Agent created successfully!


In [54]:
# ---------------------------------------------------------
# STEP 14: CREATE THE SUPERVISOR AGENT
# ---------------------------------------------------------

# Define the Supervisor Agent.
def supervisor_agent(state: ResearchState):

    # Check what information has already been produced.
    research = state["research"]
    fact_check = state["fact_check"]
    analysis = state["analysis"]
    final_report = state["final_report"]

    # Decide which agent should work next.
    if not research:
        next_agent = "researcher"

    elif not fact_check:
        next_agent = "fact_checker"

    elif not analysis:
        next_agent = "analyst"

    elif not final_report:
        next_agent = "writer"

    else:
        next_agent = "writer"

    # Return the Supervisor's decision.
    return {
        "next_agent": next_agent
    }


# Confirm that the Supervisor was created.
print("Supervisor Agent created successfully!")

Supervisor Agent created successfully!


In [55]:
# ---------------------------------------------------------
# STEP 15: CREATE THE ROUTING FUNCTION
# ---------------------------------------------------------

# Define the function that reads the Supervisor's decision.
def route_from_supervisor(state: ResearchState):

    # Get the next agent selected by the Supervisor.
    next_agent = state["next_agent"]

    # Return the name of the next node in the graph.
    return next_agent


# Confirm that the routing function was created.
print("Routing function created successfully!")

Routing function created successfully!


In [56]:
# ---------------------------------------------------------
# STEP 16: CREATE THE LANGGRAPH WORKFLOW
# ---------------------------------------------------------

# Create a StateGraph using our shared ResearchState.
builder = StateGraph(ResearchState)


# Add the Supervisor to the workflow.
builder.add_node("supervisor", supervisor_agent)

# Add the Research Agent.
builder.add_node("researcher", research_agent)

# Add the Fact Checker Agent.
builder.add_node("fact_checker", fact_checker_agent)

# Add the Analysis Agent.
builder.add_node("analyst", analysis_agent)

# Add the Writer Agent.
builder.add_node("writer", writer_agent)


# Start the workflow with the Supervisor.
builder.add_edge(START, "supervisor")


# Tell LangGraph to use the Supervisor's decision
# to select the next agent.
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "researcher": "researcher",
        "fact_checker": "fact_checker",
        "analyst": "analyst",
        "writer": "writer"
    }
)


# After the Research Agent finishes,
# return to the Supervisor.
builder.add_edge("researcher", "supervisor")

# After the Fact Checker finishes,
# return to the Supervisor.
builder.add_edge("fact_checker", "supervisor")

# After the Analysis Agent finishes,
# return to the Supervisor.
builder.add_edge("analyst", "supervisor")


# The Writer produces the final report,
# so the workflow ends after the Writer.
builder.add_edge("writer", END)


# Compile the workflow.
research_graph = builder.compile()


# Confirm that the graph was created.
print("LangGraph workflow created successfully!")

LangGraph workflow created successfully!


In [57]:
# ---------------------------------------------------------
# STEP 17: CREATE THE INITIAL WORKFLOW STATE
# ---------------------------------------------------------

# Create the starting state for our research workflow.
initial_state = {

    # The question that our AI system will answer.
    "question": "What are the applications of generative AI in education?",

    # These fields start empty.
    "research": "",
    "fact_check": "",
    "analysis": "",
    "final_report": "",

    # The Supervisor will decide this later.
    "next_agent": ""
}


# Display the initial state.
print("Initial workflow state created!")

# Display the question.
print("Question:", initial_state["question"])

Initial workflow state created!
Question: What are the applications of generative AI in education?


In [58]:
# ---------------------------------------------------------
# STEP 18: TEST THE SUPERVISOR
# ---------------------------------------------------------

# Send the initial state to the Supervisor.
supervisor_result = supervisor_agent(initial_state)


# Display the Supervisor's decision.
print("===== SUPERVISOR DECISION =====")
print(supervisor_result["next_agent"])

===== SUPERVISOR DECISION =====
researcher


In [59]:
# ---------------------------------------------------------
# STEP 19: TEST THE ROUTING FUNCTION
# ---------------------------------------------------------

# Add the Supervisor's decision to our test state.
initial_state["next_agent"] = supervisor_result["next_agent"]


# Ask the routing function where the workflow should go.
next_node = route_from_supervisor(initial_state)


# Display the next node.
print("===== NEXT WORKFLOW NODE =====")
print(next_node)

===== NEXT WORKFLOW NODE =====
researcher


In [60]:
# ---------------------------------------------------------
# STEP 20: TEST SUPERVISOR AFTER RESEARCH
# ---------------------------------------------------------

# Create a state where research has already been completed.
state_after_research = {

    # Original question.
    "question": "What are the applications of generative AI in education?",

    # Pretend that research has been completed.
    "research": "Generative AI can help with personalized learning, content creation, tutoring, and feedback.",

    # Fact checking has not been completed yet.
    "fact_check": "",

    # Analysis has not been completed yet.
    "analysis": "",

    # Final report has not been completed yet.
    "final_report": "",

    # Supervisor will update this field.
    "next_agent": ""
}


# Run the Supervisor.
result = supervisor_agent(state_after_research)


# Display the Supervisor's decision.
print("===== SUPERVISOR DECISION =====")
print(result["next_agent"])

===== SUPERVISOR DECISION =====
fact_checker


In [61]:
# ---------------------------------------------------------
# STEP 21: TEST SUPERVISOR AFTER FACT CHECKING
# ---------------------------------------------------------

# Create a state where research and fact checking are complete.
state_after_fact_check = {

    # Original question.
    "question": "What are the applications of generative AI in education?",

    # Research has been completed.
    "research": "Generative AI can help with personalized learning.",

    # Fact checking has been completed.
    "fact_check": "The main claims appear reasonable, but sources should be verified.",

    # Analysis is still missing.
    "analysis": "",

    # Final report is still missing.
    "final_report": "",

    # Supervisor will update this field.
    "next_agent": ""
}


# Run the Supervisor.
result = supervisor_agent(state_after_fact_check)


# Display the decision.
print("===== SUPERVISOR DECISION =====")
print(result["next_agent"])

===== SUPERVISOR DECISION =====
analyst


In [62]:
# ---------------------------------------------------------
# STEP 22: TEST SUPERVISOR BEFORE WRITING
# ---------------------------------------------------------

# Create a state where research, fact checking,
# and analysis are all complete.
state_before_writing = {

    # Original question.
    "question": "What are the applications of generative AI in education?",

    # Research is complete.
    "research": "Generative AI can support personalized learning.",

    # Fact checking is complete.
    "fact_check": "The claims appear reasonable but require source verification.",

    # Analysis is complete.
    "analysis": "Generative AI can improve personalization and support teachers.",

    # The final report has not been created yet.
    "final_report": "",

    # Supervisor will update this field.
    "next_agent": ""
}


# Run the Supervisor.
result = supervisor_agent(state_before_writing)


# Display the decision.
print("===== SUPERVISOR DECISION =====")
print(result["next_agent"])

===== SUPERVISOR DECISION =====
writer


In [63]:
# ---------------------------------------------------------
# STEP 23: CREATE A RESPONSE TEXT HELPER
# ---------------------------------------------------------

# Create a helper function to safely extract text
# from a Gemini response.
def get_response_text(response):

    # Get the content returned by Gemini.
    content = response.content

    # If Gemini returned a list of content blocks,
    # process each block separately.
    if isinstance(content, list):

        # Create an empty list to store text parts.
        text_parts = []

        # Go through each content block.
        for block in content:

            # If the block is a dictionary,
            # try to get its "text" value.
            if isinstance(block, dict):
                text_parts.append(block.get("text", ""))

            # Otherwise convert the block to text.
            else:
                text_parts.append(str(block))

        # Join all text parts together.
        return " ".join(text_parts).strip()

    # If content is already normal text,
    # simply convert it to a string.
    return str(content).strip()


# Confirm that the helper was created.
print("Response text helper created successfully!")

Response text helper created successfully!


In [64]:
# ---------------------------------------------------------
# STEP 24: UPDATE THE RESEARCH AGENT
# ---------------------------------------------------------

# Redefine the Research Agent using the response helper.
def research_agent(state: ResearchState):

    # Get the user's question from the shared state.
    question = state["question"]

    # Create instructions for Gemini.
    prompt = f"""
You are the Research Agent in a multi-agent research system.

Research this question:

{question}

Provide:
1. Key facts
2. Important explanations
3. Relevant details
4. Useful examples

Keep the information concise.

Do not write the final report.
"""

    # Send the research instructions to Gemini.
    response = llm.invoke(prompt)

    # Safely extract the response text.
    research_text = get_response_text(response)

    # Return the research result.
    return {
        "research": research_text
    }


# Confirm that the Research Agent was updated.
print("Research Agent updated successfully!")

Research Agent updated successfully!


In [65]:
# ---------------------------------------------------------
# STEP 25: UPDATE THE FACT CHECKER AND ANALYSIS AGENTS
# ---------------------------------------------------------

# ---------------- FACT CHECKER AGENT ----------------

# Redefine the Fact Checker Agent.
def fact_checker_agent(state: ResearchState):

    # Get the research information.
    research = str(state["research"])[:6000]

    # Create fact-checking instructions.
    prompt = f"""
You are the Fact Checker Agent.

Review this research:

{research}

Identify:
1. Reliable-looking claims
2. Claims needing verification
3. Possible inaccuracies
4. Important limitations

Keep the response concise.
"""

    # Send the request to Gemini.
    response = llm.invoke(prompt)

    # Safely extract the response text.
    fact_check_text = get_response_text(response)

    # Return the fact-checking result.
    return {
        "fact_check": fact_check_text
    }


# ---------------- ANALYSIS AGENT ----------------

# Redefine the Analysis Agent.
def analysis_agent(state: ResearchState):

    # Get the research.
    research = str(state["research"])[:5000]

    # Get the fact-checking information.
    fact_check = str(state["fact_check"])[:3000]

    # Create analysis instructions.
    prompt = f"""
You are the Analysis Agent.

Analyze the information below.

RESEARCH:
{research}

FACT CHECK:
{fact_check}

Provide:
1. Main insight
2. Important conclusion
3. Practical implication
4. Limitation

Keep the analysis concise.
"""

    # Send the request to Gemini.
    response = llm.invoke(prompt)

    # Safely extract the response text.
    analysis_text = get_response_text(response)

    # Return the analysis result.
    return {
        "analysis": analysis_text
    }


# Confirm both agents were updated.
print("Fact Checker Agent updated successfully!")
print("Analysis Agent updated successfully!")

Fact Checker Agent updated successfully!
Analysis Agent updated successfully!


In [66]:
# ---------------------------------------------------------
# STEP 26: UPDATE THE WRITER AGENT
# ---------------------------------------------------------

# Redefine the Writer Agent using the response helper.
def writer_agent(state: ResearchState):

    # Get the original question.
    question = state["question"]

    # Get the research information.
    research = str(state["research"])[:5000]

    # Get the fact-checking information.
    fact_check = str(state["fact_check"])[:3000]

    # Get the analysis.
    analysis = str(state["analysis"])[:3000]

    # Create instructions for the Writer Agent.
    prompt = f"""
You are the Writer Agent in a multi-agent research system.

Write a clear final report answering the question.

QUESTION:
{question}

RESEARCH:
{research}

FACT CHECK:
{fact_check}

ANALYSIS:
{analysis}

Structure the report as:

1. Title
2. Introduction
3. Main Findings
4. Analysis
5. Conclusion

Use simple, professional language.

Do not mention the internal agents.
"""

    # Send the writing request to Gemini.
    response = llm.invoke(prompt)

    # Safely extract the response text.
    final_report_text = get_response_text(response)

    # Return the final report.
    return {
        "final_report": final_report_text
    }


# Confirm that the Writer Agent was updated.
print("Writer Agent updated successfully!")

Writer Agent updated successfully!


In [67]:
# ---------------------------------------------------------
# STEP 27: REBUILD THE LANGGRAPH WORKFLOW
# ---------------------------------------------------------

# Create a new StateGraph using our ResearchState.
builder = StateGraph(ResearchState)


# Add the Supervisor node.
builder.add_node("supervisor", supervisor_agent)

# Add the Research Agent node.
builder.add_node("researcher", research_agent)

# Add the Fact Checker Agent node.
builder.add_node("fact_checker", fact_checker_agent)

# Add the Analysis Agent node.
builder.add_node("analyst", analysis_agent)

# Add the Writer Agent node.
builder.add_node("writer", writer_agent)


# Start the workflow with the Supervisor.
builder.add_edge(START, "supervisor")


# Use the Supervisor's decision to select
# the next agent.
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "researcher": "researcher",
        "fact_checker": "fact_checker",
        "analyst": "analyst",
        "writer": "writer"
    }
)


# Return to the Supervisor after research.
builder.add_edge("researcher", "supervisor")

# Return to the Supervisor after fact checking.
builder.add_edge("fact_checker", "supervisor")

# Return to the Supervisor after analysis.
builder.add_edge("analyst", "supervisor")


# End the workflow after the Writer.
builder.add_edge("writer", END)


# Compile the graph.
research_graph = builder.compile()


# Confirm that the graph compiled successfully.
print("LangGraph workflow rebuilt successfully!")

LangGraph workflow rebuilt successfully!


In [68]:
# ---------------------------------------------------------
# STEP 28: DISPLAY THE WORKFLOW GRAPH
# ---------------------------------------------------------

# Display the graph as Mermaid text.
print(research_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	fact_checker(fact_checker)
	analyst(analyst)
	writer(writer)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analyst --> supervisor;
	fact_checker --> supervisor;
	researcher --> supervisor;
	supervisor -.-> analyst;
	supervisor -.-> fact_checker;
	supervisor -.-> researcher;
	supervisor -.-> writer;
	writer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [70]:
# ---------------------------------------------------------
# STEP 29: CREATE A COMPLETE TEST STATE
# ---------------------------------------------------------

# Create a sample state containing information
# for every stage of the workflow.
complete_test_state = {

    # Original user question.
    "question": "What are the applications of generative AI in education?",

    # Sample research information.
    "research": "Generative AI can support personalized learning, content creation, tutoring, and automated feedback.",

    # Sample fact-checking information.
    "fact_check": "The main claims are reasonable, but individual claims should be verified with reliable sources.",

    # Sample analysis.
    "analysis": "Generative AI can improve personalized learning and reduce some routine tasks for teachers, but human oversight remains important.",

    # Sample final report.
    "final_report": "",

    # Supervisor will decide the next step.
    "next_agent": ""
}


# Confirm that the complete test state was created.
print("Complete test state created successfully!")

Complete test state created successfully!


In [71]:
# ---------------------------------------------------------
# STEP 30: TEST THE FINAL SUPERVISOR DECISION
# ---------------------------------------------------------

# Run the Supervisor using the complete test state.
final_supervisor_result = supervisor_agent(complete_test_state)


# Display the Supervisor's decision.
print("===== FINAL SUPERVISOR DECISION =====")
print(final_supervisor_result["next_agent"])

===== FINAL SUPERVISOR DECISION =====
writer


In [72]:
# ---------------------------------------------------------
# STEP 31: TEST FINAL ROUTING
# ---------------------------------------------------------

# Add the Supervisor's decision to the test state.
complete_test_state["next_agent"] = final_supervisor_result["next_agent"]


# Use the routing function to determine
# the next workflow node.
final_next_node = route_from_supervisor(complete_test_state)


# Display the next node.
print("===== FINAL ROUTING RESULT =====")
print(final_next_node)

===== FINAL ROUTING RESULT =====
writer


In [73]:
# ---------------------------------------------------------
# STEP 32: CHECK THE GRAPH NODES
# ---------------------------------------------------------

# Get the graph structure.
graph_structure = research_graph.get_graph()

# Display all nodes in the workflow.
print("===== GRAPH NODES =====")

for node_name in graph_structure.nodes:
    print("-", node_name)

===== GRAPH NODES =====
- __start__
- supervisor
- researcher
- fact_checker
- analyst
- writer
- __end__


In [74]:
# ---------------------------------------------------------
# STEP 33: CHECK THE GRAPH CONNECTIONS
# ---------------------------------------------------------

# Display all connections between workflow nodes.
print("===== GRAPH CONNECTIONS =====")

for edge in graph_structure.edges:
    print(edge.source, "→", edge.target)

===== GRAPH CONNECTIONS =====
__start__ → supervisor
analyst → supervisor
fact_checker → supervisor
researcher → supervisor
supervisor → analyst
supervisor → fact_checker
supervisor → researcher
supervisor → writer
writer → __end__


In [75]:
# ---------------------------------------------------------
# STEP 34: CREATE THE FINAL INPUT STATE
# ---------------------------------------------------------

# Create the initial state that will be given
# to the complete LangGraph workflow.
final_input = {

    # The research question our agents will answer.
    "question": "What are the applications of generative AI in education?",

    # These fields begin empty.
    "research": "",
    "fact_check": "",
    "analysis": "",
    "final_report": "",

    # The Supervisor will select the first agent.
    "next_agent": ""
}


# Confirm that the final input is ready.
print("Final workflow input created successfully!")

# Display the question.
print("Question:", final_input["question"])

Final workflow input created successfully!
Question: What are the applications of generative AI in education?


In [76]:
# ---------------------------------------------------------
# STEP 35: CREATE AN EXECUTION LOG
# ---------------------------------------------------------

# Create a list to store the order in which
# our agents are executed.
execution_log = []


# Create a function to record an agent's name.
def log_agent(agent_name):

    # Add the agent name to the execution log.
    execution_log.append(agent_name)


# Confirm that the execution log was created.
print("Execution log created successfully!")

Execution log created successfully!


In [77]:
# ---------------------------------------------------------
# STEP 35: CREATE AN EXECUTION LOG
# ---------------------------------------------------------

# Create a list to store the order in which
# our agents are executed.
execution_log = []


# Create a function to record an agent's name.
def log_agent(agent_name):

    # Add the agent name to the execution log.
    execution_log.append(agent_name)


# Confirm that the execution log was created.
print("Execution log created successfully!")

Execution log created successfully!


In [78]:
# ---------------------------------------------------------
# STEP 36: CREATE A WORKFLOW SUMMARY
# ---------------------------------------------------------

# Store the names of all agents in our system.
agent_names = [
    "Supervisor",
    "Research Agent",
    "Fact Checker Agent",
    "Analysis Agent",
    "Writer Agent"
]


# Display the agents.
print("===== MULTI-AGENT SYSTEM =====")

for agent in agent_names:
    print("-", agent)


# Display the purpose of the project.
print("\nProject: AI Research & Report Generation")
print("Framework: LangGraph")
print("Language: Python")
print("LLM: Google Gemini")

===== MULTI-AGENT SYSTEM =====
- Supervisor
- Research Agent
- Fact Checker Agent
- Analysis Agent
- Writer Agent

Project: AI Research & Report Generation
Framework: LangGraph
Language: Python
LLM: Google Gemini


In [79]:
# ---------------------------------------------------------
# STEP 37: TEST THE WORKFLOW ENTRY POINT
# ---------------------------------------------------------

# Start with our final input state.
workflow_test_state = final_input.copy()


# Run the Supervisor.
supervisor_test = supervisor_agent(workflow_test_state)


# Store the Supervisor's decision.
workflow_test_state["next_agent"] = supervisor_test["next_agent"]


# Find the next workflow node.
first_node = route_from_supervisor(workflow_test_state)


# Display the result.
print("===== WORKFLOW ENTRY TEST =====")
print("Starting node: supervisor")
print("Next node:", first_node)

===== WORKFLOW ENTRY TEST =====
Starting node: supervisor
Next node: researcher


In [81]:
# ---------------------------------------------------------
# STEP 38: CREATE A STATE VALIDATOR
# ---------------------------------------------------------

# This function checks whether our workflow state
# contains all the required fields.
def validate_state(state: ResearchState):

    # List all fields that our workflow needs.
    required_fields = [
        "question",
        "research",
        "fact_check",
        "analysis",
        "final_report",
        "next_agent"
    ]

    # Find any fields that are missing.
    missing_fields = [
        field for field in required_fields
        if field not in state
    ]

    # If there are missing fields, the state is not valid.
    if missing_fields:
        return False, missing_fields

    # If nothing is missing, the state is valid.
    return True, []


# Test the validator using our final_input.
is_valid, missing = validate_state(final_input)

# Display the result.
print("State is valid:", is_valid)

# Display missing fields if there are any.
print("Missing fields:", missing)

State is valid: True
Missing fields: []


In [82]:
# ---------------------------------------------------------
# STEP 39: CREATE REQUIREMENTS.TXT
# ---------------------------------------------------------

# Create a requirements file containing the packages
# needed to run our LangGraph project.
requirements = """
langgraph
langchain
langchain-google-genai
python-dotenv
"""

# Write the package list to requirements.txt.
with open("requirements.txt", "w") as file:
    file.write(requirements.strip())

# Confirm that the file was created.
print("requirements.txt created successfully!")

requirements.txt created successfully!


In [83]:
# ---------------------------------------------------------
# STEP 40: CREATE .GITIGNORE
# ---------------------------------------------------------

# These files and folders should NOT be uploaded to GitHub.
gitignore_content = """
.env
__pycache__/
*.pyc
.ipynb_checkpoints/
"""

# Create the .gitignore file.
with open(".gitignore", "w") as file:
    file.write(gitignore_content.strip())

# Confirm that the file was created.
print(".gitignore created successfully!")

.gitignore created successfully!


In [84]:
# ---------------------------------------------------------
# STEP 41: CHECK PROJECT FILES
# ---------------------------------------------------------

# Import os so we can work with files and folders.
import os

# Get the list of files in the current project folder.
files = os.listdir()

# Display the files.
print("Files in the project folder:")

for file in files:
    print("-", file)

Files in the project folder:
- .env
- .gitignore
- .ipynb_checkpoints
- multi_agent_workflow.ipynb
- requirements.txt


In [85]:
# ---------------------------------------------------------
# STEP 42: CHECK GITIGNORE PROTECTION
# ---------------------------------------------------------

# Read the contents of the .gitignore file.
with open(".gitignore", "r") as file:
    gitignore = file.read()

# Check whether .env is included.
if ".env" in gitignore:
    print("SUCCESS: .env is protected by .gitignore!")
else:
    print("WARNING: .env is NOT protected!")

SUCCESS: .env is protected by .gitignore!


In [86]:
# ---------------------------------------------------------
# STEP 43: CHECK WORKFLOW STRUCTURE
# ---------------------------------------------------------

# Display the names of the nodes in our LangGraph workflow.
print("===== WORKFLOW NODES =====")

for node in research_graph.nodes:
    print("-", node)

# Display a confirmation message.
print("\nLangGraph workflow structure is ready!")

===== WORKFLOW NODES =====
- __start__
- supervisor
- researcher
- fact_checker
- analyst
- writer

LangGraph workflow structure is ready!


In [87]:
# ---------------------------------------------------------
# STEP 44: CREATE WORKFLOW SUMMARY
# ---------------------------------------------------------

# Store the main agents and their responsibilities.
workflow_summary = {
    "Supervisor": "Controls the workflow and decides which agent runs next.",
    "Research Agent": "Collects and organizes information about the question.",
    "Fact Checker Agent": "Reviews the research for possible errors and limitations.",
    "Analysis Agent": "Analyzes the research and identifies important insights.",
    "Writer Agent": "Creates the final report using the collected information."
}

# Display the responsibilities of each agent.
print("===== MULTI-AGENT WORKFLOW =====")

for agent, responsibility in workflow_summary.items():
    print(f"\n{agent}")
    print(responsibility)

===== MULTI-AGENT WORKFLOW =====

Supervisor
Controls the workflow and decides which agent runs next.

Research Agent
Collects and organizes information about the question.

Fact Checker Agent
Reviews the research for possible errors and limitations.

Analysis Agent
Analyzes the research and identifies important insights.

Writer Agent
Creates the final report using the collected information.


In [88]:
# ---------------------------------------------------------
# STEP 45: TEST SUPERVISOR ROUTING
# ---------------------------------------------------------

# Create a state where no work has been completed.
state_1 = final_input.copy()

# Ask the Supervisor which agent should run first.
result_1 = supervisor_agent(state_1)

print("When nothing is completed:")
print("Next agent:", result_1["next_agent"])


# Create a state where research is completed.
state_2 = final_input.copy()
state_2["research"] = "Research completed."

# Ask the Supervisor for the next agent.
result_2 = supervisor_agent(state_2)

print("\nAfter research:")
print("Next agent:", result_2["next_agent"])


# Create a state where research and fact checking are completed.
state_3 = final_input.copy()
state_3["research"] = "Research completed."
state_3["fact_check"] = "Fact checking completed."

# Ask the Supervisor for the next agent.
result_3 = supervisor_agent(state_3)

print("\nAfter fact checking:")
print("Next agent:", result_3["next_agent"])

When nothing is completed:
Next agent: researcher

After research:
Next agent: fact_checker

After fact checking:
Next agent: analyst


In [89]:
# ---------------------------------------------------------
# STEP 46: SIMULATE THE COMPLETE WORKFLOW
# ---------------------------------------------------------

# Start with an empty workflow state.
simulation_state = final_input.copy()

# Store the order in which the agents should run.
simulation_order = []

# Keep checking the Supervisor's decision
# until the Writer is reached.
while True:

    # Ask the Supervisor which agent should run next.
    decision = supervisor_agent(simulation_state)

    # Get the selected agent.
    next_agent = decision["next_agent"]

    # Save the selected agent to our log.
    simulation_order.append(next_agent)

    # Simulate the agent completing its task.
    if next_agent == "researcher":
        simulation_state["research"] = "Dummy research completed."

    elif next_agent == "fact_checker":
        simulation_state["fact_check"] = "Dummy fact checking completed."

    elif next_agent == "analyst":
        simulation_state["analysis"] = "Dummy analysis completed."

    elif next_agent == "writer":
        simulation_state["final_report"] = "Dummy final report completed."
        break

# Display the simulated execution order.
print("===== SIMULATED WORKFLOW =====")

for number, agent in enumerate(simulation_order, start=1):
    print(f"{number}. {agent}")

print("\nWorkflow routing works correctly!")

===== SIMULATED WORKFLOW =====
1. researcher
2. fact_checker
3. analyst
4. writer

Workflow routing works correctly!


In [90]:
# ---------------------------------------------------------
# STEP 47: CREATE PROJECT DESCRIPTION
# ---------------------------------------------------------

# Store a short description of our project.
project_description = """
Multi-Agent Workflow with LangGraph

This project is an AI-powered research and report generation
system built using LangGraph and Gemini.

The system uses multiple specialized agents:

1. Supervisor - controls the workflow.
2. Research Agent - collects information.
3. Fact Checker Agent - reviews the information.
4. Analysis Agent - identifies important insights.
5. Writer Agent - generates the final report.

LangGraph is used to connect the agents and manage
the workflow state.
"""

# Display the project description.
print(project_description)


Multi-Agent Workflow with LangGraph

This project is an AI-powered research and report generation
system built using LangGraph and Gemini.

The system uses multiple specialized agents:

1. Supervisor - controls the workflow.
2. Research Agent - collects information.
3. Fact Checker Agent - reviews the information.
4. Analysis Agent - identifies important insights.
5. Writer Agent - generates the final report.

LangGraph is used to connect the agents and manage
the workflow state.



In [91]:
# ---------------------------------------------------------
# STEP 48: CREATE PROJECT INFORMATION
# ---------------------------------------------------------

# Store important information about the project.
project_info = {
    "Project Name": "Multi-Agent Workflow with LangGraph",
    "Language": "Python",
    "Framework": "LangGraph",
    "LLM": "Google Gemini",
    "Environment": "Jupyter Notebook + Anaconda",
    "Agents": [
        "Supervisor",
        "Research Agent",
        "Fact Checker Agent",
        "Analysis Agent",
        "Writer Agent"
    ]
}

# Display the project information.
print("===== PROJECT INFORMATION =====")

for key, value in project_info.items():
    print(f"{key}: {value}")

===== PROJECT INFORMATION =====
Project Name: Multi-Agent Workflow with LangGraph
Language: Python
Framework: LangGraph
LLM: Google Gemini
Environment: Jupyter Notebook + Anaconda
Agents: ['Supervisor', 'Research Agent', 'Fact Checker Agent', 'Analysis Agent', 'Writer Agent']


In [94]:
# ---------------------------------------------------------
# STEP 49: DISPLAY WORKFLOW DIAGRAM
# ---------------------------------------------------------

# Import the display function from IPython.
from IPython.display import display, Markdown

# Create the workflow diagram as a normal string.
workflow_diagram = """
## Multi-Agent Workflow

USER QUESTION
      |
      v
SUPERVISOR
      |
      v
RESEARCHER
      |
      v
FACT CHECKER
      |
      v
ANALYST
      |
      v
WRITER
      |
      v
FINAL REPORT
"""

# Display the workflow diagram in the notebook.
display(Markdown(workflow_diagram))


## Multi-Agent Workflow

USER QUESTION
      |
      v
SUPERVISOR
      |
      v
RESEARCHER
      |
      v
FACT CHECKER
      |
      v
ANALYST
      |
      v
WRITER
      |
      v
FINAL REPORT


In [95]:
# ---------------------------------------------------------
# STEP 50: CREATE README FILE
# ---------------------------------------------------------

# Store the README content.
readme_content = """
# Multi-Agent Workflow with LangGraph

## Project Overview

This project demonstrates a multi-agent AI workflow built
using LangGraph and Google Gemini.

The system receives a user question and passes it through
multiple specialized agents. Each agent performs a specific
task before the final report is generated.

## Workflow

User Question
     |
     v
Supervisor
     |
     v
Research Agent
     |
     v
Fact Checker Agent
     |
     v
Analysis Agent
     |
     v
Writer Agent
     |
     v
Final Report

## Agents

### 1. Supervisor
Controls the workflow and decides which agent should run next.

### 2. Research Agent
Collects important information related to the user's question.

### 3. Fact Checker Agent
Reviews the research and identifies claims that may require
verification or have limitations.

### 4. Analysis Agent
Analyzes the research and identifies important insights,
conclusions, and practical implications.

### 5. Writer Agent
Combines the research, fact checking, and analysis to create
the final report.

## Technologies

- Python
- LangGraph
- LangChain
- Google Gemini
- python-dotenv
- Jupyter Notebook
- Anaconda

## Project Structure

multi-agent-langgraph/
|
|-- multi_agent_workflow.ipynb
|-- README.md
|-- requirements.txt
|-- .env
|-- .gitignore

## How It Works

1. The user provides a research question.
2. The Supervisor checks the current workflow state.
3. The Research Agent gathers information.
4. The Fact Checker Agent reviews the research.
5. The Analysis Agent analyzes the information.
6. The Writer Agent creates the final report.
7. LangGraph manages the workflow and shared state.

## Security

The Gemini API key is stored in a `.env` file.

The `.env` file is excluded from GitHub using `.gitignore`.

Never publish the API key in the source code or GitHub repository.

## Example Question

What are the applications of generative AI in education?

## Current Status

The LangGraph workflow, agents, routing logic, state
management, and project documentation have been implemented.

The final AI execution requires an available Gemini API quota.

## Future Improvements

- Add web-search tools for real-time research.
- Add source citations.
- Improve fact verification.
- Add parallel agent execution.
- Add a user interface.
- Add persistent conversation memory.
"""

# Write the README content to README.md.
with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content.strip())

# Confirm that the README was created.
print("README.md created successfully!")

README.md created successfully!


In [96]:
# ---------------------------------------------------------
# STEP 51: CHECK README CONTENT
# ---------------------------------------------------------

# Open the README file and read its contents.
with open("README.md", "r", encoding="utf-8") as file:
    readme = file.read()

# Display the README.
print(readme)

# Multi-Agent Workflow with LangGraph

## Project Overview

This project demonstrates a multi-agent AI workflow built
using LangGraph and Google Gemini.

The system receives a user question and passes it through
multiple specialized agents. Each agent performs a specific
task before the final report is generated.

## Workflow

User Question
     |
     v
Supervisor
     |
     v
Research Agent
     |
     v
Fact Checker Agent
     |
     v
Analysis Agent
     |
     v
Writer Agent
     |
     v
Final Report

## Agents

### 1. Supervisor
Controls the workflow and decides which agent should run next.

### 2. Research Agent
Collects important information related to the user's question.

### 3. Fact Checker Agent
Reviews the research and identifies claims that may require
verification or have limitations.

### 4. Analysis Agent
Analyzes the research and identifies important insights,
conclusions, and practical implications.

### 5. Writer Agent
Combines the research, fact checking, and ana

In [97]:
# ---------------------------------------------------------
# STEP 52: FINAL PROJECT FILE CHECK
# ---------------------------------------------------------

# Import os so we can inspect the project folder.
import os

# Define the files that our project should contain.
required_files = [
    "multi_agent_workflow.ipynb",
    "README.md",
    "requirements.txt",
    ".gitignore",
    ".env"
]

# Check each required file.
print("===== PROJECT FILE CHECK =====")

for file in required_files:

    # Check whether the file exists.
    if os.path.exists(file):
        print(f"✓ {file}")
    else:
        print(f"✗ {file} - NOT FOUND")

===== PROJECT FILE CHECK =====
✓ multi_agent_workflow.ipynb
✓ README.md
✓ requirements.txt
✓ .gitignore
✓ .env


In [98]:
# ---------------------------------------------------------
# STEP 53: CHECK GIT INSTALLATION
# ---------------------------------------------------------

# Import subprocess so Python can run a Git command.
import subprocess

# Ask Git for its installed version.
result = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True
)

# Display the installed Git version.
print(result.stdout)

git version 2.38.0.windows.1



In [99]:
# ---------------------------------------------------------
# STEP 54: CHECK GIT REPOSITORY STATUS
# ---------------------------------------------------------

# Run the Git status command.
result = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

# Display the result.
print(result.stdout)

# Display any error message if Git reports one.
if result.stderr:
    print(result.stderr)

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.env.txt
	../.ipynb_checkpoints/
	../AgenticX-Offer-Letter.pdf
	../Book2.pdf
	../DPR-NASEERA_AUGUST_2025.xlsx
	../DPR-NASEERA__SEPTEMBER_2025.xlsx
	../DPR-NASEERA_october_2025.xlsx
	../Fractions.pptx
	../GATE_Question Paper.docx
	../GATE_Question Paper.pdf
	../NASEERA NM.pdf
	../Screenshot 2026-08-29 180534.png
	../adibe.docx
	../autorun.inf
	../azim files/
	../desktop.ini
	../geo.pptx
	../icons.pptx
	../int.docx
	./
	../oppo/
	../tool-using-research-agent/
	../~$DPR_NASEERA _MARCH-2025.xlsx
	../~$cbt--data struture.pptx

nothing added to commit but untracked files present (use "git add" to track)



In [100]:
# ---------------------------------------------------------
# STEP 55: CHECK CURRENT PROJECT FOLDER
# ---------------------------------------------------------

# Import os so we can see the folder where Jupyter is running.
import os

# Display the current working folder.
print("Current folder:")
print(os.getcwd())

Current folder:
C:\Users\USER\Desktop\multi-agent-langgraph


In [101]:
# ---------------------------------------------------------
# STEP 55: REMOVE THE INCORRECT GIT REPOSITORY
# ---------------------------------------------------------

# Import os and shutil for folder management.
import os
import shutil

# The incorrect Git repository is one level above
# our current project folder.
parent_git_folder = os.path.abspath(os.path.join("..", ".git"))

# Check whether that Git repository exists.
if os.path.exists(parent_git_folder):

    # Remove ONLY the .git folder.
    # This does NOT delete your PDFs, Excel files,
    # PowerPoints, or other personal files.
    shutil.rmtree(parent_git_folder)

    print("Incorrect Desktop Git repository removed safely!")

else:
    print("No incorrect Git repository found in the parent folder.")

Incorrect Desktop Git repository removed safely!


In [102]:
# ---------------------------------------------------------
# STEP 56: INITIALIZE GIT IN THE PROJECT FOLDER
# ---------------------------------------------------------

# Initialize Git in the current multi-agent-langgraph folder.
result = subprocess.run(
    ["git", "init"],
    capture_output=True,
    text=True
)

# Display Git's response.
print(result.stdout)

# Display any error message.
if result.stderr:
    print(result.stderr)

Initialized empty Git repository in C:/Users/USER/Desktop/multi-agent-langgraph/.git/



In [103]:
# ---------------------------------------------------------
# STEP 57: CHECK THE CORRECT GIT REPOSITORY
# ---------------------------------------------------------

# Check the Git status of our project.
result = subprocess.run(
    ["git", "status"],
    capture_output=True,
    text=True
)

# Display the Git status.
print(result.stdout)

# Display errors, if any.
if result.stderr:
    print(result.stderr)

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	README.md
	multi_agent_workflow.ipynb
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)



In [ ]:
# ---------------------------------------------------------
# STEP 58: ADD PROJECT FILES TO GIT
# ---------------------------------------------------------

# Add all files in the current project folder to Git.
# The .gitignore file will prevent .env from being added.
result = subprocess.run(
    ["git", "add", "."],
    capture_output=True,
    text=True
)

# Display any Git message.
print(result.stdout)

# Display any error message.
if result.stderr:
    print(result.stderr)

print("Project files added to Git!")
